# P6｜BEATs Shared Temporal Package

**状态：Scientific HOLD；8/20 core。** 目的：候选 HF 主线使用 BEATs frame tokens、token_mask、source-time time_map 与 temporal head，同时维持三条 non-HF native lanes。

## 唯一变量、匹配对照与四数据集边界

matched comparator=P2，但 P6 改变了 HF encoder/head 与时间接口，因此只能作为 package-level comparison；不能声称纯 encoder 或纯 head attribution。必须先通过 non-HF pooled-parity gate，证明 ICBHI/SPRSound/KAUH 路径与 P2 一致。

- ICBHI cycle flat4、SPRSound event binary/raw7、KAUH recording raw9 继续 pooled/native；KAUH B/D/E 同 patient group。
- HF 以共享 token 时间区间对齐四个 native temporal channels：I、E、CAS、DAS；每通道独立保留 observation_mask 与 valid_mask。raw conservative policy 中 missing/unknown/gap 全部 masked；P6 执行候选冻结为 source-paper-native one-vs-rest rasterization，interval 外的零仅是 source-task constructed negative，不是 raw negative、shared normal 或 P8 shared evidence。
- 输入为 16 kHz source-time-lineaged windows；统一 encoder 输出为 tokens float [B,L,D]、token_mask bool [B,L]、time_map float [B,L,2]（source seconds, half-open）、pooled float [B,D]；HF temporal logits 固定为 [B,L,4]，对应 I/E/CAS/DAS。

In [ ]:
from pathlib import Path
import os

TEMPORAL_CONTRACT = {
    "tokens": ["B", "L", "D"],
    "token_mask": ["B", "L"],
    "time_map": ["B", "L", 2],
    "pooled": ["B", "D"],
    "hf_channels": ["I", "E", "CAS", "DAS"],
    "hf_logits": ["B", "L", 4],
    "hf_observation_mask": ["B", "L", 4],
    "hf_valid_mask": ["B", "L", 4],
    "time_unit": "source_seconds_half_open",
}
PIPELINE = {
    "id": "P6",
    "comparator": "P2_package_level",
    "seed": 20260728,
    "split_policy": "reuse_P2_immutable_receipts",
    "encoder": "BEATs_shared_temporal_adapter",
    "hf_head": "minimal_temporal_head_pending_freeze",
    "hf_target_policy": "PAPER_NATIVE_RASTERIZED_OVR",
    "hf_alignment": "token_center_in_interval",
    "hf_negative_semantics": "source_task_constructed_not_raw_normal",
    "hf_shared_label_eligible": False,
    "hf_source_policy_reference": "docs/datasets/four_dataset_task_contract_review_2026-07-28.md",
    "non_hf_gate": "pooled_parity_with_P2",
    "contract_modules": ["baseline.multidataset_pipeline.contracts", "baseline.multidataset_pipeline.beats_temporal", "baseline.multidataset_pipeline.hf_data"],
    "engineering_tests": ["tests/test_multidataset_pipeline.py::BEATsTemporalContractTest", "tests/test_hf_data.py::HFDataContractTest"],
    "read_only_verifier_entry": "python -m baseline.multidataset_pipeline.verify_hf_contract --phase all",
    "update_budget": None,
    "selection": None,
    "output_dir": "result/reproduce/P6_beats_shared_temporal",
    "receipt_path": "result/reproduce/P6_beats_shared_temporal/P6_receipt.json",
}
PROJECT_ROOT = Path(os.environ.get("ACOUSTIC_PROJECT_ROOT", Path.cwd())).resolve()
APPROVAL_RECEIPT = os.environ.get("P6_APPROVAL_RECEIPT")


## 科学 gate 与运行前审批

执行前需通过 schema/shape/dtype、显式 batch/model device parity、BEATs token-mask 展平、source-time 单调性、zero-padding invariance、短长音频/crop、HF 四通道 token-center alignment、逐通道 observation/valid mask、raw conservative 与 paper-native constructed-negative 边界、empty/missing regression、non-HF pooled parity、gradient scope、resource profile 与 independent verifier。还须冻结 checkpoint/SHA、input adapter、head、budget、selection、metrics 和 approval。任何 gate 未通过均 HOLD。

In [ ]:
required = [PIPELINE["update_budget"], PIPELINE["selection"], APPROVAL_RECEIPT]
if any(value in (None, "") for value in required):
    raise RuntimeError("P6 HOLD: temporal contract, parity, budget, selection, and approval must be verified")
approval_path = PROJECT_ROOT / APPROVAL_RECEIPT
if not approval_path.is_file():
    raise FileNotFoundError(approval_path)
DRY_RUN_PLAN = {"pipeline": PIPELINE, "contract": TEMPORAL_CONTRACT, "execute": False}


## 输出、receipt 与结果表

receipt 必须含 temporal schema version、device、token/mask/time-map geometry、I/E/CAS/DAS channel order、target policy、alignment=token_center_in_interval、negative_semantics=source_task_constructed_not_raw_normal、shared_label_eligible=false、逐通道 positive/constructed-negative/masked counts、source-policy reference、checkpoint/frontend hashes、window/crop lineage、padding verifier、non-HF parity、native/time metrics、seed/updates/selection 与 warnings。

| Gate / comparison | Result | Decision |
|---|---:|---|
| non-HF P6↔P2 parity；HF package P6↔P2 | Not run | HOLD |

**Test Result = Not run。Decision = HOLD。Claim boundary：P6−P2 是 package-level effect，不是纯 encoder/head attribution。**